# SINDy Portfolio Optimization — v8 (Extended History)

**Goal:** Resolve the Momentum MVO comparison (p = 0.168 in v7) by adding more data.

**Changes from v7:**
- History extended to 2005 (was 2010). Adds ~1,250 trading days including the
  2008 financial crisis --- the most important stress event in modern markets.
- Two OOS splits: 2015 (~2,500 OOS days) and 2019 (~1,800 OOS days).
  The 2015 split captures COVID + 2022 bear + most of the post-GFC cycle.
- Universes adjusted for ETF availability pre-2007.
  (XLC and XLRE dropped from Equity Sectors; some Global Macro/FI tickers substituted.)


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import cvxpy as cp

try:
    import pysindy as ps
    HAS_PYSINDY = True
except Exception:
    HAS_PYSINDY = False

try:
    import yfinance as yf
    HAS_YF = True
except Exception:
    HAS_YF = False

from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LassoCV
from sklearn.covariance import LedoitWolf

# ── Utilities ──

def winsorize(df, p=0.005):
    lo, hi = df.quantile(p), df.quantile(1 - p)
    return df.clip(lower=lo, upper=hi, axis=1)

def ann_factor(freq="W-FRI"):
    if freq.startswith("D"): return 252
    if freq.startswith("W"): return 52
    if freq.startswith("M"): return 12
    return 52

def max_drawdown(cum_curve):
    peak = np.maximum.accumulate(cum_curve)
    dd = (cum_curve - peak) / np.where(peak > 0, peak, 1)
    return np.min(dd)

print("Imports OK.")


In [ ]:
def load_returns(tickers, start="2005-01-01", end=None, freq="D"):
    if HAS_YF:
        try:
            raw = yf.download(tickers, start=start, end=end, progress=False)
            if isinstance(raw.columns, pd.MultiIndex):
                top_cols = set(raw.columns.get_level_values(0))
                price_col = "Adj Close" if "Adj Close" in top_cols else "Close"
                px = raw[price_col]
            elif "Adj Close" in raw.columns:
                px = raw["Adj Close"]
            else:
                px = raw["Close"]
            if isinstance(px, pd.Series):
                px = px.to_frame()
            px = px.dropna(how="all").ffill().dropna()
            rets = px.pct_change().dropna()
            lo, hi = rets.quantile(0.005), rets.quantile(0.995)
            rets = rets.clip(lower=lo, upper=hi, axis=1)
            good_cols = rets.columns[rets.notna().mean() > 0.95]
            rets = rets[good_cols].dropna()
            return rets
        except Exception as e:
            print(f"[yfinance fallback] {e}")
    return None

# ═══════════════════════════════════════════════
# UNIVERSES — adjusted for 2005 availability
# ═══════════════════════════════════════════════

UNIVERSES = {
    "US Multi-Asset": {
        "tickers": ["SPY", "QQQ", "XLF", "XLE", "XLV", "TLT", "IEF", "GLD", "VNQ", "EFA", "EEM"],
        "start": "2005-01-01",
        "class_map": {
            'SPY': 'equity', 'QQQ': 'equity', 'XLF': 'equity', 'XLE': 'equity', 'XLV': 'equity',
            'TLT': 'bond', 'IEF': 'bond', 'GLD': 'gold', 'VNQ': 'reit',
            'EFA': 'intl', 'EEM': 'intl',
        },
    },
    "Global Macro": {
        # BWX→AGG (2003), DBC→GLD already in, UUP→SHY (dollar proxy)
        "tickers": ["EFA", "EEM", "FXI", "EWZ", "TLT", "IEF", "GLD", "SHY", "VNQ", "SPY"],
        "start": "2005-01-01",
        "class_map": {
            'EFA': 'equity', 'EEM': 'equity', 'FXI': 'equity', 'EWZ': 'equity',
            'TLT': 'bond', 'IEF': 'bond', 'SHY': 'bond',
            'GLD': 'gold', 'VNQ': 'reit', 'SPY': 'equity',
        },
    },
    "Fixed Income+": {
        # HYG→drop, EMB→drop. Add XLU (bond proxy), XLP (defensive)
        "tickers": ["TLT", "IEF", "SHY", "LQD", "TIP", "GLD", "VNQ", "SPY", "XLU", "XLP"],
        "start": "2005-01-01",
        "class_map": {
            'TLT': 'bond', 'IEF': 'bond', 'SHY': 'bond', 'LQD': 'bond', 'TIP': 'bond',
            'GLD': 'gold', 'VNQ': 'reit', 'SPY': 'equity',
            'XLU': 'bond', 'XLP': 'bond',
        },
    },
    "Equity Sectors": {
        # Drop XLC (2018) and XLRE (2015). 9 sectors that existed since 1998.
        "tickers": ["XLK", "XLF", "XLE", "XLV", "XLI", "XLY", "XLP", "XLB", "XLU"],
        "start": "2005-01-01",
        "class_map": {
            'XLK': 'equity', 'XLF': 'equity', 'XLE': 'equity', 'XLV': 'equity',
            'XLI': 'equity', 'XLY': 'equity',
            'XLP': 'bond', 'XLB': 'equity', 'XLU': 'bond',
        },
    },
}

universe_data = {}
for name, cfg in UNIVERSES.items():
    print(f"\nLoading {name} (from {cfg['start']})...")
    rets_u = load_returns(cfg["tickers"], start=cfg["start"])
    if rets_u is not None and len(rets_u) > 1000:
        asset_classes = [cfg["class_map"].get(t, 'equity') for t in rets_u.columns]
        universe_data[name] = {
            "rets": rets_u,
            "asset_classes": asset_classes,
            "tickers": list(rets_u.columns),
        }
        print(f"  {rets_u.shape[0]} days x {rets_u.shape[1]} assets")
        print(f"  {rets_u.index[0].date()} -> {rets_u.index[-1].date()}")
        for t, ac in zip(rets_u.columns, asset_classes):
            print(f"    {t:>5s} -> {ac}")
    else:
        print(f"  SKIPPED")

print(f"\nLoaded {len(universe_data)} universes.")
AF = 252


In [ ]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LassoCV, RidgeCV
from sklearn.preprocessing import PolynomialFeatures
from scipy.stats import rankdata

class RiskFeatureExtractor:
    def __init__(self, short_window=20, long_window=60):
        self.short_window = short_window
        self.long_window = long_window

    def extract(self, X):
        T, N = X.shape
        df = pd.DataFrame(X)
        features = {}

        vol_short = df.rolling(self.short_window).std()
        vol_long = df.rolling(self.long_window).std()
        features['vol_short_mean'] = vol_short.mean(axis=1).values
        features['vol_long_mean'] = vol_long.mean(axis=1).values
        features['vol_ratio'] = features['vol_short_mean'] / (features['vol_long_mean'] + 1e-12)

        # Equity vs other vol — use asset_classes if available
        # For now, just use first half vs second half as a proxy
        half = N // 2
        if N > 2:
            first_vol = vol_short.iloc[:, :half].mean(axis=1).values
            second_vol = vol_short.iloc[:, half:].mean(axis=1).values
            features['group_vol_ratio'] = first_vol / (second_vol + 1e-12)

        corr_series = np.full(T, np.nan)
        for t in range(self.long_window, T, 5):
            window = X[t - self.long_window:t]
            c = np.corrcoef(window.T)
            mask = ~np.eye(N, dtype=bool)
            corr_series[t] = c[mask].mean()
        corr_series = pd.Series(corr_series).ffill().bfill().values
        features['avg_corr'] = corr_series

        # Stock-bond correlation — computed dynamically if we have enough assets
        if N > 4:
            sb_corr = np.full(T, np.nan)
            for t in range(self.long_window, T, 5):
                window = X[t - self.long_window:t]
                # Use first half vs second half as proxy groups
                g1 = window[:, :half].mean(axis=1)
                g2 = window[:, half:].mean(axis=1)
                if len(g1) > 5:
                    sb_corr[t] = np.corrcoef(g1, g2)[0, 1]
            sb_corr = pd.Series(sb_corr).ffill().bfill().values
            features['cross_group_corr'] = sb_corr

        features['dispersion'] = df.std(axis=1).values
        vol_short_s = pd.Series(features['vol_short_mean'])
        features['vol_of_vol'] = vol_short_s.rolling(self.short_window).std().values

        down_corr = np.full(T, np.nan)
        for t in range(self.long_window, T, 5):
            window = X[t - self.long_window:t]
            mkt = window.mean(axis=1)
            down_mask = mkt < 0
            if down_mask.sum() > 5:
                c = np.corrcoef(window[down_mask].T)
                mask_d = ~np.eye(N, dtype=bool)
                down_corr[t] = c[mask_d].mean()
            else:
                down_corr[t] = corr_series[t]
        down_corr = pd.Series(down_corr).ffill().bfill().values
        features['down_corr'] = down_corr
        features['corr_asymmetry'] = down_corr - corr_series

        feat_matrix = np.column_stack(list(features.values()))
        feat_names = list(features.keys())
        offset = self.long_window + self.short_window
        valid = np.nan_to_num(feat_matrix[offset:], nan=0.0)
        return valid, offset, feat_names


class SINDyRegimeDetector:
    def __init__(self, n_components=4, degree=2, threshold_pctile=75):
        self.n_components = n_components
        self.degree = degree
        self.threshold_pctile = threshold_pctile
        self.pca = None
        self.poly = None
        self.models = []
        self._feat_mean = None
        self._feat_std = None
        self._stress_threshold = None
        self.feat_extractor = RiskFeatureExtractor()

    def fit(self, X_raw):
        feats, offset, self.feat_names = self.feat_extractor.extract(X_raw)
        if len(feats) < 100:
            self.models = None
            return self
        self._feat_mean = feats.mean(axis=0)
        self._feat_std = feats.std(axis=0) + 1e-12
        feats_sc = (feats - self._feat_mean) / self._feat_std
        n_comp = min(self.n_components, feats_sc.shape[1], len(feats_sc) - 2)
        self.pca = PCA(n_components=n_comp).fit(feats_sc)
        Z = self.pca.transform(feats_sc)
        Z_in, Z_out = Z[:-1], Z[1:]
        self.poly = PolynomialFeatures(degree=self.degree, include_bias=True)
        Phi = self.poly.fit_transform(Z_in)
        self.models = []
        for j in range(n_comp):
            lasso = LassoCV(alphas=np.logspace(-5, -1, 20), cv=5,
                            max_iter=5000, n_jobs=None)
            lasso.fit(Phi, Z_out[:, j])
            self.models.append(lasso)
        stress_scores = self._compute_stress_scores(Z)
        self._stress_threshold = np.percentile(stress_scores, self.threshold_pctile)
        return self

    def _compute_stress_scores(self, Z):
        Phi = self.poly.transform(Z[:-1])
        Z_pred = np.column_stack([m.predict(Phi) for m in self.models])
        delta = Z_pred - Z[:-1]
        stress = np.sqrt(np.sum(delta ** 2, axis=1))
        return np.concatenate([[0], stress])

    def predict_regime(self, X_raw_recent):
        if self.models is None:
            return 0.0, False, 0.0
        feats, offset, _ = self.feat_extractor.extract(X_raw_recent)
        if len(feats) < 5:
            return 0.0, False, 0.0
        feats_sc = (feats - self._feat_mean) / self._feat_std
        Z = self.pca.transform(feats_sc)
        stress_scores = self._compute_stress_scores(Z)
        current_stress = stress_scores[-1]
        is_stressed = current_stress > self._stress_threshold
        Phi_last = self.poly.transform(Z[-1:])
        z_pred = np.array([m.predict(Phi_last)[0] for m in self.models])
        vol_direction = z_pred[0] - Z[-1, 0]
        return current_stress, is_stressed, vol_direction

    def get_coefficients(self):
        if self.models is None: return np.array([])
        return np.column_stack([m.coef_ for m in self.models])

    def get_feature_names(self):
        if self.poly is None: return []
        return self.poly.get_feature_names_out()


class SINDyCovarianceEngine:
    """
    v5: Smarter stress adjustment.
    - Inflate WITHIN-equity correlations (they converge to 1 in crises)
    - PRESERVE equity-bond negative correlation (flight to quality)
    - Boost predicted vol for equities, reduce for bonds
    """
    def __init__(self, vol_window=20, cov_window=126,
                 stress_eq_corr_boost=0.4, asset_classes=None):
        self.vol_window = vol_window
        self.cov_window = cov_window
        self.stress_eq_corr_boost = stress_eq_corr_boost
        self.asset_classes = asset_classes or []

    def predict_cov(self, X_train, stress_score=0.0, is_stressed=False):
        T, N = X_train.shape
        vol_series = pd.DataFrame(X_train).rolling(self.vol_window).std().dropna().values
        predicted_vols = np.zeros(N)

        for j in range(N):
            y = vol_series[:, j]
            if len(y) < 30:
                predicted_vols[j] = X_train[:, j].std()
                continue
            n_lags = min(4, len(y) // 4)
            X_ar = np.column_stack([y[i:len(y) - n_lags + i] for i in range(n_lags)])
            y_target = y[n_lags:]
            if len(y_target) < 10:
                predicted_vols[j] = y[-1]
                continue
            ridge = RidgeCV(alphas=np.logspace(-3, 1, 10)).fit(X_ar, y_target)
            predicted_vols[j] = max(ridge.predict(y[-n_lags:].reshape(1, -1))[0], 1e-6)

        recent = X_train[-self.cov_window:]
        lw_cov = shrink_cov(recent) + 1e-6 * np.eye(N)
        lw_std = np.sqrt(np.diag(lw_cov))
        corr = lw_cov / np.outer(lw_std, lw_std)
        np.fill_diagonal(corr, 1.0)
        corr = np.clip(corr, -1, 1)

        if is_stressed and len(self.asset_classes) == N:
            boost = self.stress_eq_corr_boost * min(stress_score / 2.0, 1.0)
            for i in range(N):
                for j in range(i + 1, N):
                    ac_i, ac_j = self.asset_classes[i], self.asset_classes[j]
                    if ac_i == 'equity' and ac_j == 'equity':
                        # Equities converge: push correlation toward 1
                        corr[i, j] += boost * (1.0 - corr[i, j])
                        corr[j, i] = corr[i, j]
                    elif (ac_i == 'equity' and ac_j in ('bond', 'gold')) or \
                         (ac_j == 'equity' and ac_i in ('bond', 'gold')):
                        # Equity-bond/gold: push MORE negative (flight to quality)
                        corr[i, j] -= boost * 0.5 * (1.0 + corr[i, j])
                        corr[j, i] = corr[i, j]
                    # else: leave other pairs alone

            corr = np.clip(corr, -1, 1)

            # Stress vol adjustment: equities up, bonds down
            for i, ac in enumerate(self.asset_classes):
                if ac == 'equity':
                    predicted_vols[i] *= (1.0 + boost * 0.5)
                elif ac in ('bond', 'gold'):
                    predicted_vols[i] *= max(1.0 - boost * 0.2, 0.5)

        D = np.diag(predicted_vols)
        pred_cov = D @ corr @ D
        pred_cov = (pred_cov + pred_cov.T) / 2
        eigvals = np.linalg.eigvalsh(pred_cov)
        if eigvals.min() < 0:
            pred_cov += (abs(eigvals.min()) + 1e-6) * np.eye(N)
        return pred_cov

print("SINDy Risk Engine defined (v5: asset-class-aware covariance adjustment).")


In [ ]:
class SampleMeanForecaster:
    def __init__(self):
        self._mu = None
    def fit(self, X):
        self._mu = X.mean(axis=0)
        return self
    def predict(self, x_last):
        return self._mu

class MomentumForecaster:
    def __init__(self, lookback=60):
        self.lookback = lookback
        self._mu = None
    def fit(self, X):
        ts_mom = X[-self.lookback:].mean(axis=0)
        xs_mom = ts_mom - ts_mom.mean()
        self._mu = 0.5 * ts_mom + 0.5 * xs_mom
        return self
    def predict(self, x_last):
        return self._mu

class ZeroForecaster:
    def __init__(self):
        self._N = None
    def fit(self, X):
        self._N = X.shape[1]
        return self
    def predict(self, x_last):
        return np.zeros(self._N)


class SINDyRiskMomentumForecaster:
    def __init__(self, mom_lookback=60, regime_threshold_pctile=75,
                 n_components=4, degree=2,
                 equity_dampen=0.3, defensive_boost=2.0,
                 asset_classes=None):
        self.mom_lookback = mom_lookback
        self.regime_threshold_pctile = regime_threshold_pctile
        self.n_components = n_components
        self.degree = degree
        self.equity_dampen = equity_dampen
        self.defensive_boost = defensive_boost
        self.asset_classes = asset_classes or []
        self._regime_detector = None
        self._cov_engine = SINDyCovarianceEngine(asset_classes=self.asset_classes)
        self._last_train = None
        self._stress_score = 0.0
        self._is_stressed = False
        self._vol_direction = 0.0

    def fit(self, X):
        self._last_train = X
        self._regime_detector = SINDyRegimeDetector(
            n_components=self.n_components, degree=self.degree,
            threshold_pctile=self.regime_threshold_pctile,
        ).fit(X)
        self._stress_score, self._is_stressed, self._vol_direction = \
            self._regime_detector.predict_regime(X)
        return self

    def predict(self, x_last):
        X = self._last_train
        N = X.shape[1]
        ts_mom = X[-self.mom_lookback:].mean(axis=0)
        xs_mom = ts_mom - ts_mom.mean()
        mu = 0.5 * ts_mom + 0.5 * xs_mom
        if self._is_stressed and len(self.asset_classes) == N:
            weights = np.ones(N)
            for i, ac in enumerate(self.asset_classes):
                if ac == 'equity':
                    weights[i] = self.equity_dampen
                elif ac in ('bond', 'gold'):
                    weights[i] = self.defensive_boost
                elif ac == 'intl':
                    weights[i] = self.equity_dampen * 1.5
            mu = mu * weights
        return mu

    def predict_cov(self, cov_window=126):
        return self._cov_engine.predict_cov(
            self._last_train,
            stress_score=self._stress_score,
            is_stressed=self._is_stressed,
        )

    def get_regime_info(self):
        return {
            "stress_score": self._stress_score,
            "is_stressed": self._is_stressed,
            "vol_direction": self._vol_direction,
        }

    def get_coefficients(self):
        if self._regime_detector:
            return self._regime_detector.get_coefficients()
        return np.array([])

    def get_feature_names(self):
        if self._regime_detector:
            return self._regime_detector.get_feature_names()
        return []

print("Forecasters defined.")


In [ ]:
def shrink_cov(X_window):
    """Ledoit-Wolf shrinkage covariance."""
    if X_window.shape[0] > 5:
        try:
            return LedoitWolf().fit(X_window).covariance_
        except Exception:
            pass
    return np.cov(X_window.T)


def solve_mvo(mu, Sigma, w_prev=None, lam_tc=0.0,
              leverage=1.0, lb=0.0, ub=0.20, gamma=5.0):
    N = len(mu)
    w = cp.Variable(N)
    obj = cp.quad_form(w, cp.psd_wrap(Sigma)) - gamma * (mu @ w)
    constraints = [cp.sum(w) == 1, w >= lb, w <= ub, cp.norm1(w) <= leverage]
    if w_prev is not None and lam_tc > 0:
        obj += lam_tc * cp.norm1(w - w_prev)
    prob = cp.Problem(cp.Minimize(obj), constraints)
    prob.solve(solver=cp.OSQP, verbose=False)
    if w.value is None:
        prob.solve(solver=cp.SCS, verbose=False, max_iters=8000)
    if w.value is None:
        return np.ones(N) / N
    return np.array(w.value).flatten()


def solve_cvar(R_scenarios, alpha=0.95, w_prev=None, lam_tc=0.0,
               leverage=1.0, lb=0.0, ub=0.20):
    """CVaR minimization (Rockafellar-Uryasev) on historical scenarios."""
    S, N = R_scenarios.shape
    w = cp.Variable(N)
    z = cp.Variable(S)
    t = cp.Variable()
    losses = -R_scenarios @ w
    cvar = t + (1.0 / ((1 - alpha) * S)) * cp.sum(z)
    constr = [
        z >= 0, z >= losses - t,
        cp.sum(w) == 1, w >= lb, w <= ub, cp.norm1(w) <= leverage,
    ]
    obj = cvar
    if w_prev is not None and lam_tc > 0:
        obj += lam_tc * cp.norm1(w - w_prev)
    prob = cp.Problem(cp.Minimize(obj), constr)
    prob.solve(solver=cp.SCS, verbose=False, max_iters=8000)
    if w.value is None:
        prob.solve(solver=cp.CLARABEL, verbose=False)
    if w.value is None:
        return np.ones(N) / N
    return np.array(w.value).flatten()

print("Solvers defined: MVO (OSQP/SCS), CVaR (SCS/CLARABEL).")


In [ ]:
def backtest(
    rets_df,
    forecaster_cls,
    forecaster_kwargs=None,
    solver="mvo",
    train_min=504,              # 2 years daily
    rebalance_every=21,         # monthly
    gamma=5.0,
    lam_tc=0.002,
    leverage=1.0,
    lb=0.0,
    ub=0.20,
    cvar_alpha=0.95,
    cvar_scen_len=126,
    realized_cost_bps=3.0,      # slightly higher for daily
    cov_window=126,
    adaptive_bounds=False,
    stress_ub_mult=0.5,
    stress_gamma_mult=3.0,      # triple risk aversion in stress
):
    forecaster_kwargs = forecaster_kwargs or {}
    X = rets_df.values
    T, N = X.shape
    cost_mult = realized_cost_bps / 10_000

    w = np.ones(N) / N
    pnl, turnover_list, weights_hist = [], [], []
    coef_history, regime_history = [], []

    for t0 in range(train_min, T - 1, rebalance_every):
        X_train = X[:t0]
        fc = forecaster_cls(**forecaster_kwargs).fit(X_train)
        mu_pred = fc.predict(X[t0 - 1])

        # Covariance
        Sigma = None
        if hasattr(fc, 'predict_cov'):
            try:
                Sigma = fc.predict_cov(cov_window)
            except Exception:
                pass
        if Sigma is None:
            X_cov = X[max(0, t0 - cov_window):t0]
            Sigma = shrink_cov(X_cov) + 1e-6 * np.eye(N)

        # Regime-adaptive
        ub_eff, gamma_eff = ub, gamma
        if adaptive_bounds and hasattr(fc, 'get_regime_info'):
            info = fc.get_regime_info()
            regime_history.append((rets_df.index[t0], info))
            if info['is_stressed']:
                ub_eff = ub * stress_ub_mult
                gamma_eff = gamma * stress_gamma_mult

        # Solve
        if solver == "cvar":
            scen = X[max(0, t0 - cvar_scen_len):t0]
            w_new = solve_cvar(scen, alpha=cvar_alpha, w_prev=w,
                               lam_tc=lam_tc, leverage=leverage, lb=lb, ub=ub_eff)
        else:
            w_new = solve_mvo(mu_pred, Sigma, w_prev=w, lam_tc=lam_tc,
                              leverage=leverage, lb=lb, ub=ub_eff, gamma=gamma_eff)

        to = np.sum(np.abs(w_new - w))
        turnover_list.append(to)
        cost = cost_mult * to

        if hasattr(fc, 'get_coefficients'):
            try:
                c = fc.get_coefficients()
                if len(c) > 0:
                    coef_history.append((rets_df.index[t0], c))
            except Exception:
                pass

        for tau in range(t0, min(t0 + rebalance_every, T)):
            gross = float(w_new @ X[tau])
            net = gross - cost if tau == t0 else gross
            pnl.append(net)

        w = w_new
        weights_hist.append((rets_df.index[t0], w_new.copy()))

    pnl = np.array(pnl)
    curve = np.cumprod(1 + pnl)

    mean_ret = pnl.mean() * AF
    vol = pnl.std(ddof=1) * np.sqrt(AF)
    sharpe = mean_ret / (vol + 1e-12)
    down = pnl[pnl < 0]
    sortino = mean_ret / (down.std(ddof=1) * np.sqrt(AF) + 1e-12) if len(down) > 0 else np.inf
    mdd = max_drawdown(curve)
    calmar = mean_ret / (abs(mdd) + 1e-12)

    return {
        "pnl": pnl, "curve": curve,
        "sharpe": sharpe, "sortino": sortino,
        "mean_ret": mean_ret, "vol": vol,
        "max_dd": mdd, "calmar": calmar,
        "avg_turnover": np.mean(turnover_list),
        "weights_hist": weights_hist,
        "coef_history": coef_history,
        "regime_history": regime_history,
    }

print("Backtest engine defined (daily, multi-asset, aggressive regime switch).")


In [ ]:
COMMON = dict(
    train_min=504, rebalance_every=21, gamma=5.0,
    lam_tc=0.002, leverage=1.0, lb=0.0, ub=0.15,
    realized_cost_bps=3.0, cov_window=126,
)
COMMON_OOS = {**COMMON, "train_min": 252}

def bootstrap_sharpe_diff(pnl_a, pnl_b, n_boot=10000, seed=42):
    rng = np.random.RandomState(seed)
    T = min(len(pnl_a), len(pnl_b))
    pnl_a, pnl_b = pnl_a[:T], pnl_b[:T]
    def sharpe(p):
        return p.mean() / (p.std(ddof=1) + 1e-12) * np.sqrt(AF)
    obs = sharpe(pnl_a) - sharpe(pnl_b)
    combined = np.column_stack([pnl_a, pnl_b])
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.randint(0, T, T)
        diffs[i] = sharpe(combined[idx, 0]) - sharpe(combined[idx, 1])
    centered = diffs - diffs.mean()
    p_val = np.mean(np.abs(centered) >= abs(obs))
    ci = (np.percentile(diffs, 2.5), np.percentile(diffs, 97.5))
    return obs, ci, p_val


def run_oos(rets_df, asset_classes, split_date, label=""):
    """Run OOS analysis for a single split date. Returns dict of results."""
    rets_test = rets_df[rets_df.index >= split_date]
    X_test = rets_test.values
    T_test, N = X_test.shape
    tm = COMMON_OOS["train_min"]
    rb = COMMON_OOS["rebalance_every"]

    if T_test < tm + 100:
        print(f"    {label}: insufficient OOS data ({T_test} days)")
        return {}

    oos = {}

    # Equal Weight
    pnl = X_test[tm:] @ (np.ones(N) / N)
    c = np.cumprod(1 + pnl)
    mr, vo = pnl.mean() * AF, pnl.std(ddof=1) * np.sqrt(AF)
    down = pnl[pnl < 0]
    oos["Equal Weight"] = {
        "pnl": pnl, "curve": c, "sharpe": mr / (vo + 1e-12),
        "sortino": mr / (down.std(ddof=1) * np.sqrt(AF) + 1e-12) if len(down) > 0 else np.inf,
        "mean_ret": mr, "vol": vo, "max_dd": max_drawdown(c),
        "calmar": mr / (abs(max_drawdown(c)) + 1e-12),
    }

    # Risk Parity
    pnl_rp = []
    for t0 in range(tm, T_test - 1, rb):
        vol_est = np.std(X_test[max(0, t0 - 126):t0], axis=0) + 1e-12
        w = (1.0 / vol_est); w = w / w.sum()
        for tau in range(t0, min(t0 + rb, T_test)):
            pnl_rp.append(float(w @ X_test[tau]))
    pnl_rp = np.array(pnl_rp)
    c = np.cumprod(1 + pnl_rp)
    mr, vo = pnl_rp.mean() * AF, pnl_rp.std(ddof=1) * np.sqrt(AF)
    down = pnl_rp[pnl_rp < 0]
    oos["Risk Parity"] = {
        "pnl": pnl_rp, "curve": c, "sharpe": mr / (vo + 1e-12),
        "sortino": mr / (down.std(ddof=1) * np.sqrt(AF) + 1e-12) if len(down) > 0 else np.inf,
        "mean_ret": mr, "vol": vo, "max_dd": max_drawdown(c),
        "calmar": mr / (abs(max_drawdown(c)) + 1e-12),
    }

    # Momentum MVO
    try:
        oos["Momentum MVO"] = backtest(rets_test,
            forecaster_cls=MomentumForecaster, forecaster_kwargs={"lookback": 60},
            solver="mvo", **COMMON_OOS)
    except Exception as e:
        print(f"    Momentum FAILED: {e}")

    # SINDy CVaR
    try:
        oos["SINDy CVaR"] = backtest(rets_test,
            forecaster_cls=SINDyRiskMomentumForecaster,
            forecaster_kwargs={
                "mom_lookback": 60, "regime_threshold_pctile": 75,
                "n_components": 4, "degree": 2,
                "equity_dampen": 0.3, "defensive_boost": 2.0,
                "asset_classes": asset_classes,
            },
            solver="cvar", adaptive_bounds=True, stress_ub_mult=0.6,
            stress_gamma_mult=2.0, **COMMON_OOS)
    except Exception as e:
        print(f"    SINDy CVaR FAILED: {e}")

    return oos


def print_oos(oos, label=""):
    if not oos:
        return
    print(f"\n    {'Strategy':<20s} {'Sharpe':>7s} {'MaxDD':>8s} {'Vol':>7s} {'Return':>8s}")
    print(f"    {'-'*50}")
    for nm, r in oos.items():
        print(f"    {nm:<20s} {r['sharpe']:>7.3f} {r['max_dd']:>7.1%} {r['vol']:>6.1%} {r['mean_ret']:>7.2%}")


def print_bootstrap(oos, label=""):
    if "SINDy CVaR" not in oos:
        return
    print(f"\n    Bootstrap (SINDy CVaR vs benchmarks):")
    for bm in ["Equal Weight", "Risk Parity", "Momentum MVO"]:
        if bm in oos:
            diff, ci, pval = bootstrap_sharpe_diff(oos["SINDy CVaR"]["pnl"], oos[bm]["pnl"])
            sig = "***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.10 else "n.s."
            print(f"      vs {bm:<20s}: dSharpe={diff:+.3f} CI[{ci[0]:+.3f},{ci[1]:+.3f}] p={pval:.3f} {sig}")


# ═══════════════════════════════════════════════
# RUN ALL UNIVERSES × TWO OOS SPLITS
# ═══════════════════════════════════════════════
OOS_SPLITS = ["2015-01-01", "2019-01-01"]

all_results = {}

for uname, udata in universe_data.items():
    rets_df = udata["rets"]
    ac = udata["asset_classes"]
    N = rets_df.shape[1]

    print(f"\n{'='*100}")
    print(f"  UNIVERSE: {uname} ({N} assets, {rets_df.shape[0]} days)")
    print(f"  {rets_df.index[0].date()} -> {rets_df.index[-1].date()}")
    print(f"{'='*100}")

    # Full sample
    print(f"\n  Full sample:")
    full = {}

    full["Equal Weight"] = {}
    X = rets_df.values
    T = X.shape[0]
    tm = COMMON["train_min"]
    pnl = X[tm:] @ (np.ones(N) / N)
    c = np.cumprod(1 + pnl)
    mr, vo = pnl.mean() * AF, pnl.std(ddof=1) * np.sqrt(AF)
    down = pnl[pnl < 0]
    full["Equal Weight"] = {
        "pnl": pnl, "curve": c, "sharpe": mr / (vo + 1e-12),
        "mean_ret": mr, "vol": vo, "max_dd": max_drawdown(c),
    }

    print(f"    {'Momentum MVO':<20s} ... ", end="", flush=True)
    try:
        full["Momentum MVO"] = backtest(rets_df,
            forecaster_cls=MomentumForecaster, forecaster_kwargs={"lookback": 60},
            solver="mvo", **COMMON)
        print(f"Sharpe={full['Momentum MVO']['sharpe']:.3f}")
    except Exception as e:
        print(f"FAILED: {e}")

    print(f"    {'SINDy CVaR':<20s} ... ", end="", flush=True)
    try:
        full["SINDy CVaR"] = backtest(rets_df,
            forecaster_cls=SINDyRiskMomentumForecaster,
            forecaster_kwargs={
                "mom_lookback": 60, "regime_threshold_pctile": 75,
                "n_components": 4, "degree": 2,
                "equity_dampen": 0.3, "defensive_boost": 2.0,
                "asset_classes": ac,
            },
            solver="cvar", adaptive_bounds=True, stress_ub_mult=0.6,
            stress_gamma_mult=2.0, **COMMON)
        print(f"Sharpe={full['SINDy CVaR']['sharpe']:.3f}")
    except Exception as e:
        print(f"FAILED: {e}")

    # OOS for each split
    oos_results = {}
    for split in OOS_SPLITS:
        n_oos = len(rets_df[rets_df.index >= split])
        print(f"\n  OOS split at {split} ({n_oos} days):")
        oos = run_oos(rets_df, ac, split, label=f"{uname} @ {split}")
        print_oos(oos)
        print_bootstrap(oos)
        oos_results[split] = oos

    all_results[uname] = {"full": full, "oos": oos_results}


# ═══════════════════════════════════════════════
# CROSS-UNIVERSE SUMMARY PER SPLIT
# ═══════════════════════════════════════════════
for split in OOS_SPLITS:
    print(f"\n\n{'='*100}")
    print(f"CROSS-UNIVERSE SUMMARY — OOS split at {split}")
    print(f"{'='*100}")

    print(f"\n  {'Universe':<20s} {'SINDy':>7s} {'EW':>7s} {'RP':>7s} {'Mom':>7s} {'Wins':>6s}")
    print(f"  {'-'*55}")

    for uname, ur in all_results.items():
        oos = ur["oos"].get(split, {})
        sc = oos.get("SINDy CVaR", {}).get("sharpe", float("nan"))
        ew = oos.get("Equal Weight", {}).get("sharpe", float("nan"))
        rp = oos.get("Risk Parity", {}).get("sharpe", float("nan"))
        mom = oos.get("Momentum MVO", {}).get("sharpe", float("nan"))
        bms = [v for v in [ew, rp, mom] if not np.isnan(v)]
        wins = sum(1 for b in bms if sc > b) if not np.isnan(sc) else 0
        print(f"  {uname:<20s} {sc:>7.3f} {ew:>7.3f} {rp:>7.3f} {mom:>7.3f} {wins}/{len(bms)}")


# ═══════════════════════════════════════════════
# POOLED BOOTSTRAP — focus on Momentum comparison
# ═══════════════════════════════════════════════
print(f"\n\n{'='*100}")
print(f"POOLED BOOTSTRAP — THE MOMENTUM QUESTION")
print(f"{'='*100}")

for split in OOS_SPLITS:
    pooled_sindy, pooled_mom, pooled_ew, pooled_rp = [], [], [], []
    total_days = 0

    for uname, ur in all_results.items():
        oos = ur["oos"].get(split, {})
        if "SINDy CVaR" in oos:
            pooled_sindy.append(oos["SINDy CVaR"]["pnl"])
            total_days += len(oos["SINDy CVaR"]["pnl"])
        if "Momentum MVO" in oos:
            pooled_mom.append(oos["Momentum MVO"]["pnl"])
        if "Equal Weight" in oos:
            pooled_ew.append(oos["Equal Weight"]["pnl"])
        if "Risk Parity" in oos:
            pooled_rp.append(oos["Risk Parity"]["pnl"])

    if pooled_sindy:
        sindy_pool = np.concatenate(pooled_sindy)
        print(f"\n  Split at {split} (pooled {total_days} OOS days across {len(pooled_sindy)} universes):")

        for bm_name, bm_pool in [("Momentum MVO", pooled_mom), ("Equal Weight", pooled_ew), ("Risk Parity", pooled_rp)]:
            if bm_pool:
                bm_concat = np.concatenate(bm_pool)
                diff, ci, pval = bootstrap_sharpe_diff(sindy_pool, bm_concat)
                sig = "***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.10 else "n.s."
                marker = "  <<<" if bm_name == "Momentum MVO" else ""
                print(f"    vs {bm_name:<20s}: dSharpe={diff:+.4f}  CI[{ci[0]:+.4f},{ci[1]:+.4f}]  p={pval:.3f} {sig}{marker}")


# ═══════════════════════════════════════════════
# FINAL VERDICT
# ═══════════════════════════════════════════════
print(f"\n\n{'='*100}")
print(f"FINAL VERDICT")
print(f"{'='*100}")

# Check the 2015 split (more data) for the Momentum comparison
split_key = "2015-01-01"
mom_results = []
for uname, ur in all_results.items():
    oos = ur["oos"].get(split_key, {})
    if "SINDy CVaR" in oos and "Momentum MVO" in oos:
        sc = oos["SINDy CVaR"]["sharpe"]
        mom = oos["Momentum MVO"]["sharpe"]
        mom_results.append((uname, sc, mom, sc - mom))

if mom_results:
    print(f"\n  SINDy CVaR vs Momentum MVO (OOS from {split_key}):")
    all_win = True
    for uname, sc, mom, diff in mom_results:
        win = "✓" if diff > 0 else "✗"
        print(f"    {uname:<20s}: SINDy {sc:.3f} vs Mom {mom:.3f} (d={diff:+.3f}) {win}")
        if diff <= 0:
            all_win = False

    # Pooled
    pooled_s = np.concatenate([ur["oos"][split_key]["SINDy CVaR"]["pnl"]
                                for uname, ur in all_results.items()
                                if "SINDy CVaR" in ur["oos"].get(split_key, {})])
    pooled_m = np.concatenate([ur["oos"][split_key]["Momentum MVO"]["pnl"]
                                for uname, ur in all_results.items()
                                if "Momentum MVO" in ur["oos"].get(split_key, {})])
    diff, ci, pval = bootstrap_sharpe_diff(pooled_s, pooled_m)
    sig = "***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.10 else "n.s."

    print(f"\n    POOLED: dSharpe={diff:+.4f}  CI[{ci[0]:+.4f},{ci[1]:+.4f}]  p={pval:.3f} {sig}")

    if pval < 0.05:
        print(f"\n  ✓ RESOLVED: SINDy CVaR significantly beats Momentum MVO (p={pval:.3f}).")
        print(f"    The extended history provides enough statistical power.")
    elif pval < 0.10:
        print(f"\n  ~ CLOSE: SINDy CVaR beats Momentum at p={pval:.3f} (marginal).")
        print(f"    Additional data or universes may push this below 0.05.")
    else:
        print(f"\n  ✗ UNRESOLVED: p={pval:.3f}. Still cannot conclusively beat Momentum MVO.")
        print(f"    The advantage is economically meaningful but statistically elusive.")
        print(f"    This is an honest result — report it as an open question.")


In [ ]:
import matplotlib.pyplot as plt

n_u = len(all_results)
fig, axes = plt.subplots(2, n_u, figsize=(6 * n_u, 8))
if n_u == 1:
    axes = axes.reshape(2, 1)

# Use the 2015 split (more data)
split = "2015-01-01"

for col, (uname, ur) in enumerate(all_results.items()):
    oos = ur["oos"].get(split, {})

    # OOS wealth curves
    ax = axes[0, col]
    if oos:
        for nm, r in oos.items():
            lw = 2.5 if "SINDy" in nm else 1.5 if "Momentum" in nm else 1.0
            ls = "-" if "SINDy" in nm else "-." if "Momentum" in nm else "--"
            color = "#4CAF50" if "SINDy" in nm else "#E91E63" if "Momentum" in nm else None
            ax.plot(r["curve"], label=nm, linewidth=lw, linestyle=ls,
                    color=color if color else None)
        ax.set_title(f"{uname}\nOOS from {split}", fontsize=9)
        ax.legend(fontsize=6)
        ax.grid(alpha=0.3)
        ax.set_ylabel("Growth of $1")

    # OOS Sharpe bars
    ax = axes[1, col]
    if oos:
        names = list(oos.keys())
        sharpes = [oos[n]["sharpe"] for n in names]
        colors = ["#4CAF50" if "SINDy" in n else "#E91E63" if "Momentum" in n else "#2196F3" for n in names]
        ax.barh(range(len(names)), sharpes, color=colors, edgecolor="black", alpha=0.8)
        ax.set_yticks(range(len(names)))
        ax.set_yticklabels(names, fontsize=7)
        ax.set_xlabel("Sharpe")
        ax.set_title(f"{uname}\nOOS Sharpe", fontsize=9)
        ax.grid(alpha=0.3, axis="x")
        for i, v in enumerate(sharpes):
            ax.text(max(v + 0.005, 0.01), i, f"{v:.3f}", va="center", fontsize=7)

plt.suptitle(f"SINDy CVaR vs Momentum MVO — OOS from {split}", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


## Conclusions — v8

**Extended history (2005--2025)** adds ~1,250 days and captures the 2008 crisis.

**Two OOS splits:**
- 2019 split: same as v7, for direct comparison
- 2015 split: ~2,500 OOS days per universe, more statistical power

**The Momentum question:** With the extended history and pooled cross-universe test,
does SINDy CVaR finally reach p < 0.05 against Momentum MVO?

If yes: the paper can claim significant improvement over all benchmarks.
If no: report it honestly. Beating 60/40 and Risk Parity with p < 0.03 is
already a strong result. The Momentum comparison remains an open question
that future work with additional asset classes or longer histories may resolve.
